# Extracting Latent Dimensions of Human Epigenetic Variation Using PCA

This notebook studies whether genome-wide epigenetic signal profiles can be summarized by a small number of orthogonal patterns. It is aimed at readers comfortable with linear algebra; biological terms are introduced as needed.

Empirical claims should be made only after running the selected Roadmap tracks. The notebook uses the same functions as the command-line workflow.

## Biological context and mathematical setup

Epigenetics concerns molecular regulation associated with DNA that does not change the DNA sequence. DNA methylation adds methyl groups to DNA; effects depend on genomic context. Histones package DNA, and marks such as H3K4me3 are associated with active promoters. These signals are not direct measurements of transcription and do not prove causal function.

A genome-wide assay yields a high-dimensional vector for each biosample. Write the sample-by-feature matrix as $X\in\mathbb{R}^{n\times p}$ and center its columns: $X_c=X-\mathbf{1}\bar{x}^{\mathsf T}$. For a unit direction $v$, projected scores are $X_cv$, with sample variance $v^{\mathsf T}Sv$ where $S=X_c^{\mathsf T}X_c/(n-1)$. Maximizing this Rayleigh quotient gives the eigenproblem $Sv=\lambda v$.

If $X_c=U\Sigma V^{\mathsf T}$, then PCA loadings are columns of $V$, scores are $U\Sigma$, and eigenvalues are $\lambda_k=\sigma_k^2/(n-1)$. Explained variance is $\lambda_k/\sum_j\lambda_j$. See `MATHEMATICAL_APPENDIX.md` for the full derivation, projection geometry, and effective dimension.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

from epigenomics_pca.analysis import run_pca_analysis
from epigenomics_pca.pca import (
    pca_covariance,
    pca_sklearn,
    pca_svd,
)
from epigenomics_pca.visualization import create_figures, write_report

DATA = Path("data")
OUTPUTS = Path("outputs")

## Roadmap data and processing

The pipeline inspects the Roadmap file index, joins available signal tracks to EID biosample metadata, and uses a curated default subset of 17 EIDs with H3K4me3. A live size check found these tracks total about 8.86 GB (8.25 GiB), before processed outputs; sizes can change. The next cell downloads the tracks and runs the full workflow. For a smaller four-sample demonstration spanning stem, blood, brain, and heart biosamples (about 2.13 GB), change `all` to:

```python
[sys.executable, "-m", "epigenomics_pca", "all", "--eids", "E003,E029,E073,E083"]
```

Tracks are summarized on approximately 100 kb chromosome windows. Unreported intervals are encoded as zero, remaining non-finite values are median-imputed, signal is transformed with `log1p`, constant features are dropped, and each feature is z-scored across samples. These choices affect PCA and should be tested for sensitivity. The four-sample demonstration has at most three nonzero PCs and is for testing the workflow, not broad biological conclusions.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "epigenomics_pca", "all"],
    check=True,
)

In [ ]:
matrix = pd.read_parquet(DATA / "processed" / "X.parquet")
metadata = pd.read_csv(
    DATA / "processed" / "sample_metadata.csv", dtype=str
).fillna("")
print(f"{matrix.shape[0]} samples x {matrix.shape[1]:,} genomic features")
display(metadata.head())
display(matrix.iloc[:5, :8])

In [ ]:
values = matrix.to_numpy(dtype=np.float64)
methods = {
    "scikit-learn": pca_sklearn(values),
    "dual covariance": pca_covariance(values),
    "thin SVD": pca_svd(values),
}
reference = methods["scikit-learn"]
reference_reconstruction = reference.scores @ reference.loadings.T
for name, result in methods.items():
    np.testing.assert_allclose(
        result.eigenvalues, reference.eigenvalues, rtol=1e-8, atol=1e-10
    )
    np.testing.assert_allclose(
        result.scores @ result.loadings.T,
        reference_reconstruction,
        rtol=1e-7,
        atol=1e-9,
    )
print("PCA methods agree on retained eigenvalues and reconstructions.")

In [ ]:
analysis = run_pca_analysis(matrix, metadata, OUTPUTS)
display(analysis["variance"].head(10))
display(analysis["reconstruction_errors"].head(10))
print("Participation-ratio effective dimension:", analysis["diagnostics"]["effective_dimension"])
display(
    analysis["associations"]
    .sort_values("eta_squared", ascending=False)
    .head(10)
)

## Interpretation and limitations

Use the scree curve and cumulative variance together; there is no universal cutoff for low dimensionality. The effective dimension summarizes spectral concentration and is not the number of components needed for a chosen reconstruction tolerance. PC names have no fixed biological identity across datasets. Metadata eta-squared is descriptive, not causal. Prominent windows require annotation and independent validation before functional claims.

PCA is linear, sensitive to scaling and outliers, and prioritizes total variance rather than biological relevance. Batch, donor variation, cell mixtures, and assay quality can dominate. This pilot uses a selected set of samples, one mark, and coarse windows; it cannot establish that all human epigenomes are low-dimensional. No rose diagram is used because genomic positions and PC scores have no meaningful circular ordering.

In [ ]:
create_figures(analysis, metadata, OUTPUTS / "figures")
write_report(analysis, metadata, OUTPUTS / "REPORT.md")
print("Figures:", OUTPUTS / "figures")
print("Data-generated report:", OUTPUTS / "REPORT.md")